<a href="https://colab.research.google.com/github/Feliz-ua/goit-np-hw-03/blob/main/hw03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1.Визначення DataFrame з тривимірними векторами з word embeddings та застосування PCA для зменшення розмірності до 3-х векторів

import pickle
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import gdown

# 1. Завантаження моделі word embeddings з Google Drive
file_id = '1281E0CDneuKdflWFBUvuyUzujpdGVImz'
url = f'https://drive.google.com/uc?id={file_id}'
output_file = 'word_embeddings_subset.p'
print("Завантаження файлу...")
gdown.download(url, output_file, quiet=False)

# Відкриваємо завантажений файл
print("\nОбробка даних...")
with open(output_file, 'rb') as f:
    word_embeddings = pickle.load(f)

# Створюємо списки для слів та векторів
words = list(word_embeddings.keys())
vectors = np.array(list(word_embeddings.values()))

# 2. Використовуємо PCA для зменшення розмірності до 3-х векторів
pca = PCA(n_components=3)
vectors_3d = pca.fit_transform(vectors)

# 3. Створюємо DataFrame
df = pd.DataFrame({
    'word': words,
    'x': vectors_3d[:, 0],
    'y': vectors_3d[:, 1],
    'z': vectors_3d[:, 2]
})

print("\nГотовий DataFrame:")
print(df.tail(50))

Завантаження файлу...


Downloading...
From: https://drive.google.com/uc?id=1281E0CDneuKdflWFBUvuyUzujpdGVImz
To: /content/word_embeddings_subset.p
100%|██████████| 309k/309k [00:00<00:00, 60.6MB/s]


Обробка даних...

Готовий DataFrame:
             word         x         y         z
193          Suva -1.342464 -0.848763 -0.890645
194       Yerevan -1.057712  1.580594  1.107399
195        Zagreb -0.263384  1.756496  0.277361
196       Bishkek -1.513941  1.361890  0.874304
197        Manama -1.213770  0.059522 -0.602088
198        Kigali -1.678399 -0.704023  0.715195
199          Riga -0.314410  1.700821  0.120806
200        Lusaka -1.808256 -1.109714  0.550852
201      Tashkent -0.771772  1.287002  0.481882
202       Nicosia -0.623296  0.935671 -0.168360
203      Valletta -0.510956  0.362901 -0.022450
204      Windhoek -1.516317 -0.556969  0.450443
205      Dominica  0.745012 -0.938147 -0.404262
206         Quito -0.760938  0.082956 -0.806365
207       Tallinn -0.502376  1.958417 -0.021230
208    Bratislava -0.471912  1.834467  0.103672
209   Tegucigalpa -1.593014 -0.165771 -0.659035
210        Skopje -0.488989  1.945863  0.836101
211      Gaborone -1.511013 -1.119713  0.502368
21

In [ ]:

# 2. Визначення функції для пошуку найближчого слова до заданого вектора та демонстрація її роботи на прикладі слова "Bratislava" та двох змінених векторів
# Функція для отримання вектора конкретного слова з DataFrame
def get_vector(word, df):
    return df[df['word'] == word][['x', 'y', 'z']].values[0]

# Функція для пошуку найближчого слова
def nearest_word(target_vector, df):
    # Перетворюємо вхідний вектор на numpy масив
    target_vector = np.array(target_vector)

    # Витягуємо всі 3D координати з датафрейму
    all_vectors = df[['x', 'y', 'z']].values

    # Знаходимо відстані від нашого вектора до всіх інших
    distances = np.linalg.norm(all_vectors - target_vector, axis=1)

    # Шукаємо індекс найменшої відстані
    nearest_idx = np.argmin(distances)

    return df.iloc[nearest_idx]['word'], distances[nearest_idx]

print("--- Пошук найближчого слова ---")

# 1. Точний збіг
vec_country = get_vector('Bratislava', df)
word, dist = find_nearest_word(vec_country, df)
print(f"Ціль: 'Bratislava'. Знайдено: '{word}', відстань: {dist:.4f}")

# 2.1. Змінений вектор (додаємо шум)
noisy_vec = vec_country + np.array([0.2, -0.02, 0.05])
word_noisy, dist_noisy = find_nearest_word(noisy_vec, df)
print(f"Ціль: 'Bratislava' + шум. Знайдено: '{word_noisy}', відстань: {dist_noisy:.4f}")

# 2.2. Змінений вектор (додаємо шум)
noisy_vec = vec_country + np.array([-0.23, 0.10, 0.26])
word_noisy, dist_noisy = find_nearest_word(noisy_vec, df)
print(f"Ціль: 'Bratislava' + шум. Знайдено: '{word_noisy}', відстань: {dist_noisy:.4f}")

: 

In [ ]:
# 3. Пошук ортогонального слова (векторний добуток)
# Функція для знаходження слова, ортогонального до пари слів
def find_orthogonal_word(word1, word2, df):
    vec1 = get_vector(word1, df)
    vec2 = get_vector(word2, df)

    # Обчислюємо векторний добуток
    cross_vec = np.cross(vec1, vec2)

    # Знаходимо слово, яке лежить найближче до отриманого перпендикуляра
    nearest_word, dist = find_nearest_word(cross_vec, df)
    return nearest_word, dist

# --- ТЕСТУВАННЯ ---
print("\n--- Ортогональні слова (Векторний добуток) ---")
pairs_cross = [('Valletta', 'Tirana'), ('Tallinn', 'Vilnius'), ('Paramaribo', 'Belmopan')]

for w1, w2 in pairs_cross:
    ortho_word, dist = find_orthogonal_word(w1, w2, df)
    print(f"Ортогональне слово до '{w1}' та '{w2}': '{ortho_word}' (відстань: {dist:.2f})")


--- Ортогональні слова (Векторний добуток) ---
Ортогональне слово до 'Valletta' та 'Tirana': 'Lisbon' (відстань: 0.42)
Ортогональне слово до 'Tallinn' та 'Vilnius': 'Iran' (відстань: 0.11)
Ортогональне слово до 'Paramaribo' та 'Belmopan': 'Qatar' (відстань: 0.62)


**Пояснення результатів виконання коду**

Використання $np.cross$ - це векторний добуток. У тривимірному просторі два вектори $\mathbf{v}_1$ та $\mathbf{v}_2$ утворюють площину. Їхній векторний добуток дає третій вектор, який є строго перпендикулярним (нормаллю) до цієї площини. Тобто, алгоритм бере вектори двох міст і створює перпендикуляр до них. Після цього функція $find\ nearest\ word'$ шукає в датасеті точку, яка лежить найближче до цього розрахованого перпендикулярного вектора.
1. Tallinn та Vilnius $\rightarrow$ Iran (відстань: 0.11)
* Tallinn: [-0.502, 1.958, -0.021]
* Vilnius: [-0.711, 1.944, 0.370]
* Вектори Таллінна та Вільнюса дуже схожі - вони лежать близько один до одного, особливо по координаті Y. Вектори майже колінеарні, їхня площина дуже вузька, а векторний добуток створює нормаль, що вказує в спецефічному напрямку. Отже вектор слова "Iran" дуже близько опинився на цій лінії (відстань всього 0,11). PCA розмістив Іран так, що його вектор майже перпендикулярним до площі векторів Талліна та Вільнюса.

2. Paramaribo та Belmopan $\rightarrow$ Qatar (відстань: 0.62)
* Paramaribo: [-0.964, -0.428, -0.547]
* Belmopan: [-0.768, -0.457, -0.559]
* В цьому випадку теж бачимо два вектори, які знаходяться дуже поруч (столиці держав Латинської та Центральної Америки згрупувалися разом після PCA) по кординатам Y та Z. Найближчим сусідом до цієї нормалі виявився "Qatar", але відстань 0.62 (найбільша з усіх) каже нам про те, що Катар лежить досить далеко від перпендикуляра. Це є "найменш поганий" збіг у датасеті.

3. Valletta та Tirana $\rightarrow$ Lisbon (відстань: 0.42)
* Valletta: [-0.510, 0.362, -0.022]
* Tirana: [-0.770, 1.507, 0.617]
* На відміну від попередніх пар, ці два вектори рознесені в просторі значно сильніше (різниця по Y і Z суттєва). Вони утворюють ширшу площину. Перпендикуляр до цієї площини вказує на зону, де PCA згрупував інші європейські міста. Лісабон опинився найближчим, хоча відстань 0.42 свідчить, що це теж не ідеальний збіг.

In [ ]:
# 4. Функція для обчислення кута між словами в градусах
def calculate_angle(word1, word2, df):
    vec1 = get_vector(word1, df)
    vec2 = get_vector(word2, df)

    # Скалярний добуток
    dot_product = np.dot(vec1, vec2)

    # Довжини векторів (норми)
    norm_v1 = np.linalg.norm(vec1)
    norm_v2 = np.linalg.norm(vec2)

    # Косинус кута
    cos_theta = dot_product / (norm_v1 * norm_v2)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    # Отримуємо кут у радіанах і переводимо у градуси
    angle_rad = np.arccos(cos_theta)
    angle_deg = np.degrees(angle_rad)

    return angle_deg

print("\n--- Кут між векторами слів ---")
pairs_angle = [
    ('Tallinn', 'Iran'),
    ('Iran', 'Vilnius'),
    ('Tallinn', 'Vilnius'),
    ('Belmopan', 'Qatar'),
    ('Paramaribo', 'Qatar'),
    ('Belmopan', 'Paramaribo'),
    ('Valletta', 'Lisbon'),
    ('Tirana', 'Lisbon'),
    ('Valletta', 'Tirana')
]

for w1, w2 in pairs_angle:
    angle = calculate_angle(w1, w2, df)
    print(f"Кут між '{w1}' та '{w2}': {angle:.2f}°")


--- Кут між векторами слів ---
Кут між 'Tallinn' та 'Iran': 92.59°
Кут між 'Iran' та 'Vilnius': 92.54°
Кут між 'Tallinn' та 'Vilnius': 12.16°
Кут між 'Belmopan' та 'Qatar': 111.76°
Кут між 'Paramaribo' та 'Qatar': 119.30°
Кут між 'Belmopan' та 'Paramaribo': 7.56°
Кут між 'Valletta' та 'Lisbon': 67.73°
Кут між 'Tirana' та 'Lisbon': 75.85°
Кут між 'Valletta' та 'Tirana': 34.88°


**Висновки:**

Розрахунок реальних кутів між векторами підтверджує попередні розрахунки:
1. Майже ідеальна ортогональність (Tallinn, Vilnius, Iran).\
Кут між Tallinn та Vilnius: 12.16° — ці два вектори майже колінеарні.  
У 3D-просторі після PCA вони утворюють дуже щільний кластер (дивляться майже в одному напрямку).\
Кути до Iran: 92.59° та 92.54° — це гарний результат. Вектор слова "Iran" знаходиться під кутом, який є майже ідеальним прямим кутом (90°) до обох країн.

2. Пустота у просторі (Belmopan, Paramaribo, Qatar).\
Кут між Belmopan та Paramaribo: 7.56° — вони майже зливаються в один вектор.
Кути до Qatar: 111.76° та 119.30° —  ці кути далекі від 90°. Вони тупі.
Тут є підтвердження попереднього розрахунку. Алгоритм згенерував вектор (під кутом 90° до площини Бельмопан-Парамарибо). Але коли він почав шукати реальні слова у цій зоні, там нічого не виявилося. Найближчим словом у всьому вашому датасеті був "Qatar", але він відхиляється від перпендикуляра на 20-30 градусів. Алгоритм видав найменш поганий результат.

3. Широка площина і гострі кути (Група Valletta, Tirana, Lisbon).\
Кут між Valletta та Tirana: 34.88° — ці вектори рознесені в просторі значно ширше. Вони формують ширшу площину для розрахунку перпендикулярного вектору.
Кути до Lisbon: 67.73° та 75.85° — це не дуже близько до 90°. Кути гострі.
Аналогічна ситуація до попередньої. Нормаль вказує в порожню ділянку простору. Слово "Lisbon" було знайдено як найближче (за євклідовою відстанню) до цієї розрахованої точки нормалі.